# Lubak Alert: fine-tune YOLOv8n and export `lubak.onnx`

**Status: an OUTLINE.** The cells run in order on a free Colab GPU, but the cells and notes marked **TODO** need a decision or data that
only the team has (the flood images above all). Nothing here has been trained: no accuracy number appears in this notebook because none
has been measured. Whatever you measure goes into [`model/RESULTS.md`](https://github.com/Kaizweeen/HACKATON/blob/main/model/RESULTS.md).

**Goal.** A YOLOv8n detector for three classes, exported to ONNX at 320 px input, small enough to run in a phone browser:

| id | class | where the data comes from |
| --- | --- | --- |
| 0 | `pothole` | RDD2022 code `D40` |
| 1 | `crack` | RDD2022 codes `D00`, `D10`, `D20` |
| 2 | `flooded_road` | **not in RDD2022**: needs flood images (TODO) |

**The class order is a contract** with the app (`HAZARD_CLASSES` in `shared/src/hazard.ts`). Do not reorder it.

**Use:** Runtime > Change runtime type > GPU, then Runtime > Run all (after you filled the TODOs).

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!pip -q install ultralytics onnx onnxruntime onnxslim
import ultralytics; ultralytics.checks()

In [ ]:
# Google Drive keeps datasets and runs between sessions. Colab VMs are wiped when the session ends.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/lubak-alert')     # TODO: pick a folder; everything below lives under it
DRIVE.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/dataset')                         # built on the fast local disk, copied back to Drive at the end
RUNS = DRIVE / 'runs'
print('Drive folder:', DRIVE)

In [ ]:
# The conversion / evaluation helpers live in the repo (model/datasets.py, model/evaluate.py, model/export_onnx.py).
!git clone --depth 1 https://github.com/Kaizweeen/HACKATON /content/HACKATON
# TODO: if the repo is private, upload those three files from the model/ folder to /content/HACKATON/model/ instead.
import sys; sys.path.insert(0, '/content/HACKATON/model')
import datasets as ds, evaluate as ev
CLASSES = ds.CLASSES            # ['pothole', 'crack', 'flooded_road']  (the contract with the app)
IMGSZ = 320                     # the app feeds the model 320 x 320; training at the same size is what makes the numbers meaningful
print(CLASSES)

## 2. Data: RDD2022 (potholes and cracks)

RDD2022 is the Road Damage Dataset released with the CRDDC2022 challenge: dashcam / phone / drone photos of roads from several countries
with Pascal VOC boxes. Find the official download from the dataset's own page (search "RDD2022 CRDDC2022", or the `sekilab/RoadDamageDetector`
repository), **check its licence and cite it**, and put the unzipped folder in Drive.

Things to know before you rely on it:
* Damage codes: `D00` longitudinal crack, `D10` transverse crack, `D20` alligator crack, `D40` pothole. Anything else is ignored here.
* The countries are Japan, India, Czech Republic, Norway, USA and China (including motorbike-mounted and drone subsets). **There is no
  Philippine subset**, so expect a domain gap on local roads. The motorbike-mounted China subset is the closest in viewpoint to a handlebar-mounted phone. Plan to add and label
  local photos or video frames from the real mount position (TODO), even a few hundred.
* Frames from one road look alike, so a random split leaks near-duplicates into validation and inflates every metric. The converter splits in
  contiguous blocks. For a number you can defend, also hold out a recording or a place completely.
* Images are mostly around 600 px; at 320 px input, thin cracks lose detail. That is a real limit of this input size, measure it.

In [ ]:
RDD_ZIP = DRIVE / 'RDD2022.zip'      # TODO: path to the file you downloaded (or point RDD_ROOT at an already unzipped folder)
RDD_ROOT = Path('/content/RDD2022')
if not RDD_ROOT.exists():
    assert RDD_ZIP.exists(), f'TODO: put the RDD2022 download at {RDD_ZIP}'
    !unzip -q "{RDD_ZIP}" -d "{RDD_ROOT}"
pairs = ds.find_voc_pairs(RDD_ROOT)
print(len(pairs), 'annotated images found')
from collections import Counter
print(Counter(img.relative_to(RDD_ROOT).parts[0] for img, _ in pairs))   # images per country / subset

In [ ]:
import shutil
shutil.rmtree(DATA, ignore_errors=True)
counts = ds.convert_rdd2022(RDD_ROOT, DATA, val_fraction=0.15, keep_negatives=True, link=False)
print(counts)

## 3. Data: flood images (TODO, the big gap)

`flooded_road` has no ready-made source in this pipeline. Decide and document where the images come from **before** training:

* **Public flood-road datasets** (for example on Roboflow Universe, Kaggle or academic releases): check each licence, and check that the labels
  mean "flooded road surface", not "flood" in general (rivers, rooftops and aerial scenes are a different problem).
* **Your own photos and video frames** from the real mount, in the real weather. The most useful data you can make, and it removes the domain gap.
* Label as **one box around the flooded stretch of road** (class 2). Use CVAT, Label Studio or Roboflow; export in **YOLO format**.
* Include **negatives**: wet-but-passable roads, puddles, reflections, shadows. They are what a flood detector must not fire on.

Put the exported YOLO folder in Drive and fold it in below. If your labeller exports class 0 = flood, `class_map={0: 2}` makes it `flooded_road`.

In [ ]:
FLOOD_DIR = DRIVE / 'flood_yolo'     # TODO: your YOLO-format export (images/ + labels/)
if FLOOD_DIR.exists():
    print(ds.add_yolo_folder(FLOOD_DIR, DATA, class_map={0: 2}, prefix='flood', val_fraction=0.15))
else:
    print('TODO: no flood data yet. Training will still run, but the model will not be able to detect flooded roads.')
yaml_path = ds.write_data_yaml(DATA)
print(open(yaml_path).read())

In [ ]:
# Read this before spending GPU time: a class with only a few dozen boxes will not learn, and train/val must not share images.
import json
print(json.dumps(ds.class_histogram(DATA), indent=2))
print('images present in both train and val (must be empty):', ds.check_no_leakage(DATA))

In [ ]:
# Look at some labelled images with their boxes. Wrong class ids or shifted boxes show up here, not in the loss curve.
import random, cv2, matplotlib.pyplot as plt
COLORS = [(230, 60, 60), (240, 170, 20), (30, 120, 235)]
sample = random.Random(1).sample(sorted((DATA / 'images' / 'train').glob('*')), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
    for line in (DATA / 'labels' / 'train' / f'{path.stem}.txt').read_text().splitlines():
        c, cx, cy, bw, bh = line.split(); c = int(c)
        x1, y1, x2, y2 = [int(v) for v in ((float(cx) - float(bw) / 2) * w, (float(cy) - float(bh) / 2) * h, (float(cx) + float(bw) / 2) * w, (float(cy) + float(bh) / 2) * h)]
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[c], 3); cv2.putText(img, CLASSES[c], (x1, max(15, y1 - 5)), 0, 0.7, COLORS[c], 2)
    ax.imshow(img); ax.set_title(path.stem[:24]); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Train

Fine-tune from COCO-pretrained `yolov8n.pt`. The settings below are reasonable starting points, **not tuned values**; change them, keep the
ones that help on YOUR validation split, and write down what you ran in RESULTS.md. Things worth trying once the first run works:
more epochs, `imgsz=416` (check the on-device speed before adopting it, the app would then need `MODEL_INPUT_SIZE` changed), and class-balanced sampling if
`flooded_road` is rare.

Road scenes are not symmetric top to bottom, so vertical flips are off; left-right flips are fine (traffic side does not change what a pothole looks like).

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data=str(DATA / 'data.yaml'), imgsz=IMGSZ, epochs=100, patience=20, batch=64, seed=0, cos_lr=True,
    fliplr=0.5, flipud=0.0, mosaic=1.0, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, scale=0.5, translate=0.1,
    project=str(RUNS), name='lubak-yolov8n-320', exist_ok=True,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'
print('best weights:', BEST)

## 5. Evaluate honestly, and pick the app's thresholds

`yolo val` gives mAP and a best-F1 point, which is useful, but the app needs something more concrete: **one confidence threshold per class** in
`app/src/confirmer.ts` (`thresholds`). Too low and the map fills with false hazards; too high and real ones are missed. Below, precision and recall are
computed at the thresholds you are choosing between, on the validation split (IoU >= 0.5). Pick the lowest threshold that reaches the precision you
decided is acceptable, per class. The confirmer then requires 3 consecutive frames on top of it.

Report only what this cell prints for your final weights. A metric from a different split, image size or an earlier run is not a result.

In [ ]:
best = YOLO(BEST)
metrics = best.val(data=str(DATA / 'data.yaml'), imgsz=IMGSZ, plots=True)
print('mAP50 per class:', dict(zip(CLASSES, [round(float(x), 3) for x in metrics.box.ap50])), '| overall mAP50:', round(float(metrics.box.map50), 3))

In [ ]:
# Precision / recall per class at several confidence thresholds, from predictions made with a LOW floor so any threshold can be applied.
val_images = sorted((DATA / 'images' / 'val').glob('*'))
rows = []
for r in best.predict(val_images, imgsz=IMGSZ, conf=0.001, iou=0.45, max_det=100, stream=True, verbose=False):
    h, w = r.orig_shape
    preds = [ev.Box(int(c), *map(float, xyxy), conf=float(s)) for xyxy, c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
    label = DATA / 'labels' / 'val' / f'{Path(r.path).stem}.txt'
    truth = [ev.yolo_line_to_box(line, w, h) for line in label.read_text().splitlines() if line.strip()]
    rows.append((preds, truth))
table = ev.precision_recall(rows, CLASSES, [0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
print(ev.format_table(table))
MIN_PRECISION = 0.7     # TODO: decide what precision you can live with (a false hazard on the map costs trust)
print({name: ev.pick_threshold(rows_, MIN_PRECISION) for name, rows_ in table.items()})

Look at the failures too, not just the totals: open `val_batch*_pred.jpg` in the run folder and the confusion matrix. Typical problems to look for:
wet asphalt and shadows called flooded road, manhole covers and patches called potholes, road markings called cracks. Add examples of what it gets wrong and retrain.

## 6. Export to ONNX (the file the app loads)

`export_onnx.py` exports at 320 px, opset 12, then checks the file against what the app expects: input `[1, 3, 320, 320]`, output `[1, 7, 2100]`,
scores in 0..1, class order `pothole, crack, flooded_road`. It refuses to produce a file that would misbehave in the browser.

In [ ]:
OUT = Path('/content/lubak.onnx')
!python /content/HACKATON/model/export_onnx.py --weights "{BEST}" --out "{OUT}"

In [ ]:
# Optional but recommended: reference outputs from Ultralytics on a few of YOUR validation photos, so the team can confirm on a laptop that the
# app's TypeScript decoding gives identical detections (npm test picks the files up automatically). See model/README.md.
import shutil
PARITY = Path('/content/parity'); shutil.rmtree(PARITY, ignore_errors=True)
sample_dir = Path('/content/parity_images'); shutil.rmtree(sample_dir, ignore_errors=True); sample_dir.mkdir()
for p in random.Random(2).sample(val_images, min(8, len(val_images))): shutil.copy(p, sample_dir)
!python /content/HACKATON/model/tools/parity_ref.py --onnx "{OUT}" --images "{sample_dir}" --out "{PARITY}" --conf 0.05
!cd /content && zip -qr parity.zip parity && ls -la parity.zip

In [ ]:
# Download both files, then follow the hand-off checklist below.
from google.colab import files
files.download(str(OUT))
files.download('/content/parity.zip')
shutil.copy(OUT, RUNS / 'lubak.onnx')    # and keep a copy in Drive

## 7. Hand-off checklist

1. Copy `lubak.onnx` to **`app/public/models/lubak.onnx`** in the repo and commit it (about 12 MB).
2. Unzip `parity.zip` into **`model/work/`** (so you get `model/work/parity/reference.json`) and run `npm test`: the decode parity test must pass.
3. Put the per-class thresholds you chose into `DEFAULT_CONFIRMER_CONFIG.thresholds` in `app/src/confirmer.ts`.
4. `npm run build`, open the app on a real phone, and note the inference time and backend (Debug screen) for each phone model you try.
5. Fill in `model/RESULTS.md`: weights hash, data counts and sources with licences, the table printed above, thresholds, phone timings, known failure modes.
6. On every phone: **Debug > Reset app cache & reload** (the old model is precached).